In [1]:
import paramiko
import stat
import posixpath

HOST = "aether.ugent.be"
PORT = 2225
USERNAME = "gleamuser"
PASSWORD = "GLEAM4#h-cel_111"

transport = paramiko.Transport((HOST, PORT))
transport.connect(username=USERNAME, password=PASSWORD)
sftp = paramiko.SFTPClient.from_transport(transport)
print("Connected:", sftp.getcwd() or sftp.normalize("."))


Connected: /


In [2]:
def walk_sftp(sftp, root):
    """Recursively yield (path, SFTPAttributes) for every entry under root."""
    for entry in sftp.listdir_attr(root):
        full_path = posixpath.join(root, entry.filename)
        yield full_path, entry
        if stat.S_ISDIR(entry.st_mode):
            yield from walk_sftp(sftp, full_path)


def search_sftp(sftp, root, pattern=None, name_contains=None):
    """Search recursively under root, optionally filtering by regex pattern or substring."""
    import re

    regex = re.compile(pattern) if pattern else None
    results = []
    for path, entry in walk_sftp(sftp, root):
        name = entry.filename
        if regex and not regex.search(name):
            continue
        if name_contains and name_contains not in name:
            continue
        results.append((path, entry))
    return results


In [3]:
# List top-level contents of the SFTP root
root = "."
for entry in sftp.listdir_attr(root):
    kind = "DIR " if stat.S_ISDIR(entry.st_mode) else "FILE"
    print(f"{kind}  {entry.filename:40s}  {entry.st_size}")


DIR   data                                      4096


In [5]:
# Example: recursively search for files matching a substring/pattern
results = search_sftp(sftp, root=".", name_contains=".nc")
for path, entry in results:
    print(path, entry.st_size)

# Close the connection when done
sftp.close()
transport.close()


./data/v4.3a/yearly/Ep/Ep_2013_GLEAM_v4.3a_YR.nc 7867373
./data/v4.3a/yearly/Ep/Ep_1992_GLEAM_v4.3a_YR.nc 7872056
./data/v4.3a/yearly/Ep/Ep_2008_GLEAM_v4.3a_YR.nc 7866291
./data/v4.3a/yearly/Ep/Ep_2017_GLEAM_v4.3a_YR.nc 7869426
./data/v4.3a/yearly/Ep/Ep_2016_GLEAM_v4.3a_YR.nc 7865446
./data/v4.3a/yearly/Ep/Ep_2022_GLEAM_v4.3a_YR.nc 7867873
./data/v4.3a/yearly/Ep/Ep_2003_GLEAM_v4.3a_YR.nc 7871185
./data/v4.3a/yearly/Ep/Ep_1982_GLEAM_v4.3a_YR.nc 7864786
./data/v4.3a/yearly/Ep/Ep_2015_GLEAM_v4.3a_YR.nc 7869934
./data/v4.3a/yearly/Ep/Ep_1994_GLEAM_v4.3a_YR.nc 7872617
./data/v4.3a/yearly/Ep/Ep_1996_GLEAM_v4.3a_YR.nc 7871350
./data/v4.3a/yearly/Ep/Ep_2012_GLEAM_v4.3a_YR.nc 7866618
./data/v4.3a/yearly/Ep/Ep_2019_GLEAM_v4.3a_YR.nc 7870197
./data/v4.3a/yearly/Ep/Ep_2018_GLEAM_v4.3a_YR.nc 7863857
./data/v4.3a/yearly/Ep/Ep_1989_GLEAM_v4.3a_YR.nc 7867980
./data/v4.3a/yearly/Ep/Ep_1981_GLEAM_v4.3a_YR.nc 7869726
./data/v4.3a/yearly/Ep/Ep_2007_GLEAM_v4.3a_YR.nc 7871688
./data/v4.3a/yearly/Ep/Ep_1988_

In [6]:
# Total volume of all .nc files under ./data/v4.3a/daily/ (and subdirectories)
transport = paramiko.Transport((HOST, PORT))
transport.connect(username=USERNAME, password=PASSWORD)
sftp = paramiko.SFTPClient.from_transport(transport)

daily_root = "./data/v4.3a/daily"
nc_files = search_sftp(sftp, root=daily_root, name_contains=".nc")

total_bytes = sum(entry.st_size for _, entry in nc_files)
total_gb = total_bytes / 1024**3

print(f"Found {len(nc_files)} .nc files")
print(f"Total volume: {total_gb:.2f} GB")

sftp.close()
transport.close()


Found 644 .nc files
Total volume: 1089.99 GB


In [7]:
1090/644

1.6925465838509317